# 08 — Prediksi satu molekul dengan model tersimpan

**File:** `08_Prediksi_Molekul.ipynb`

Kita menghubungkan input SMILES dengan fitur angka, 109 skor aroma, dan keputusan label. Sel prediksi benar-benar menjalankan inferensi model utama yang sudah dilatih.

**Input:** bundle inferensi, file model terpilih, dan satu SMILES molekul. **Hubungan tahap:** 07: model/bundle terpilih → notebook ini → penggunaan hasil; integrasi Android/ONNX masih terpisah.

**Cara menjalankan:** pilih kernel `.venv` project, lalu **Restart Kernel and Run All**. Setiap notebook memuat datanya sendiri; tidak memerlukan variabel notebook lain. Source yang tampil dibaca langsung dari fungsi dalam `KODE_ML/03_Kode/` yang di-import, bukan salinan contoh.

**Batas eksekusi:** Run All bawaan membaca hasil utama yang tersimpan dan menjalankan pemeriksaan ringan. Menampilkan definisi fungsi tidak menjalankan training di dalamnya. Hasil lama dan hasil eksekusi pemeriksaan dibedakan pada tiap blok. Training baru hanya melalui gerbang eksplisit di notebook 07, yang default-nya mati.

## Persiapan: pastikan sumber berasal dari KODE_ML

Blok ini mencari folder sumber, menambahkannya ke jalur import, dan memeriksa alamat modul. `sys.dont_write_bytecode` mencegah penulisan cache Python saat belajar. Output alamat membantu mendeteksi bila kita tidak sengaja membuka salinan project lain.

In [1]:
from pathlib import Path
import sys

lokasi = Path.cwd().resolve()
kandidat = [p for induk in [lokasi, *lokasi.parents]
            for p in (induk, induk / "KODE_ML")]
kandidat.append(Path(r"C:\Users\ACER\Documents\Marvel\Skripsi\Project Skripsi\Perfume-MultiLabel-Classifier\KODE_ML"))
KODE_ML = next((p for p in kandidat if (p / "03_Kode/runtime.py").is_file()), None)
if KODE_ML is None:
    raise FileNotFoundError("Buka notebook dari folder KODE_ML yang lengkap.")
sys.dont_write_bytecode = True
sys.path.insert(0, str(KODE_ML.parent))
sys.path.insert(0, str(KODE_ML))
from IPython.display import display, Markdown
from belajar import tampilkan_sumber, tampilkan_awal
from research import runtime
assert Path(runtime.__file__).resolve().is_relative_to(KODE_ML.resolve())
print("Source aktif:", Path(runtime.__file__).resolve().parent)

Source aktif: C:\Users\ACER\Documents\Marvel\Skripsi\Project Skripsi\Perfume-MultiLabel-Classifier\KODE_ML\03_Kode


### Temukan hasil utama dan kontrak model

main_run() mencari hasil utama terkunci, bukan memilih nomor versi tertinggi. Bundle menyimpan skema fitur, label, ambang, hash, dan environment.

In [2]:
import pandas as pd
from research import inference, results

run = results.main_run()
bundle = runtime.read(run / "inference_bundle.json")
print("Folder hasil:", run)
display(pd.DataFrame([{
    "kandidat": bundle["selected_candidate"], "label": len(bundle["labels"]),
    "jumlah_fitur": bundle["feature_spec"]["dimensions"]["input"],
    "bit_Morgan": bundle["feature_spec"]["n_bits"],
    "deskriptor": bundle["feature_spec"]["descriptors"],
    "ambang": bundle["primary_threshold"]
}]))

Folder hasil: C:\Users\ACER\Documents\Marvel\Skripsi\Project Skripsi\Perfume-MultiLabel-Classifier\.local-tools\campus-transfer-full-20260909\perfume-campus-grouped-v7-m2048-d8\perfume-campus-grouped-v7-m2048-d8\runs\perfume-five-grouped-v7-m2048-d8


,kandidat,label,jumlah_fitur,bit_Morgan,deskriptor,ambang
0,lgbm_D,109,2056,2048,"[MolWt, MolLogP, NumHDonors, NumHAcceptors, TP...",0.5


### Import dan konstanta `results.py`

ARCHIVED_RUN_ID menunjuk hasil utama. Nama versi pada path adalah identitas arsip; bukan nama metode yang harus dipakai dalam judul skripsi.

In [3]:
tampilkan_awal(results)

**Import dan konstanta aktual:** `KODE_ML/03_Kode/results.py`. SHA-256: `724762a4e03ec358000821ecddff0d86436e1983256b8b0e6b3adacdb63b5f2d`.

```python
"""Alamat hasil dikunci, bukan pencarian versi terbaru."""
from .runtime import ROOT
ARCHIVED_RUN_ID = "perfume-five-grouped-v7-m2048-d8"

```

### `results.main_run` — kode aktual

**File:** `KODE_ML/03_Kode/results.py`.

**Input:** root project. **Output:** alamat hasil utama dengan complete.json. Dua lokasi yang dikenal dicoba; jika tidak tersedia, error. Fungsi tidak otomatis training ulang atau mengganti ke hasil lain.

Sel berikut menampilkan definisi lengkap beserta lokasi dan hash sumber. Isinya dipakai oleh pipeline, bukan implementasi kedua.

In [4]:
tampilkan_sumber(results.main_run)

**Source aktual:** `KODE_ML/03_Kode/results.py`, baris 5–13. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `724762a4e03ec358000821ecddff0d86436e1983256b8b0e6b3adacdb63b5f2d`.

```python
def main_run():
    candidates = (
        ROOT / "runs" / ARCHIVED_RUN_ID,
        ROOT / ".local-tools/campus-transfer-full-20260909/perfume-campus-grouped-v7-m2048-d8/perfume-campus-grouped-v7-m2048-d8/runs" / ARCHIVED_RUN_ID,
    )
    for path in candidates:
        if (path / "complete.json").is_file():
            return path
    raise FileNotFoundError("Hasil utama tidak ditemukan. Jangan training ulang hanya untuk membaca hasil.")

```

### Import dan konstanta `inference.py`

RDKit membentuk masukan, joblib memuat classifier, NumPy menggabungkan fitur. Konstanta Morgan/deskriptor berasal dari modul fitur yang sama dengan penelitian.

In [5]:
tampilkan_awal(inference)

**Import dan konstanta aktual:** `KODE_ML/03_Kode/inference.py`. SHA-256: `a9cdd7d49ed7a308134c9d6b82b47f1cbfcd720a9c58d02511ecf78a9b432a4a`.

```python
"""Prediksi satu molekul memakai model penelitian tersimpan dan skema yang sama."""
import argparse
import json
from pathlib import Path

import joblib
import numpy as np
from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors, rdFingerprintGenerator

from . import runtime as core
from .runtime import ROOT, digest
from .features import DESCRIPTORS8, N_BITS


```

### `inference.predict` — kode aktual

**File:** `KODE_ML/03_Kode/inference.py`.

**Input:** folder run dan SMILES satu molekul. **Output:** SMILES kanonis, kandidat, ambang, 109 label/skor/keputusan terurut.

1. Verifikasi bundle, environment, urutan deskriptor, dan dimensi.
2. Parse SMILES; tolak tidak valid atau lebih dari satu fragmen.
3. Bentuk Morgan + deskriptor dalam urutan training; cast float32 menjadi satu baris.
4. Per label, cek hash model/dimensi dan ambil `predict_proba(x)[0, 1]`.
5. Skor ≥ ambang berarti keputusan positif; urutkan untuk dibaca.

Warning nama fitur LightGBM disaring spesifik karena urutan ndarray telah dikunci; skor tidak berubah. **Skor belum dikalibrasi dan bukan intensitas aroma, komposisi, atau jaminan probabilitas sensoris.**

Sel berikut menampilkan definisi lengkap beserta lokasi dan hash sumber. Isinya dipakai oleh pipeline, bukan implementasi kedua.

In [6]:
tampilkan_sumber(inference.predict)

**Source aktual:** `KODE_ML/03_Kode/inference.py`, baris 16–58. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `a9cdd7d49ed7a308134c9d6b82b47f1cbfcd720a9c58d02511ecf78a9b432a4a`.

```python
def predict(run, smiles):
    run = Path(run)
    bundle = core.read(run/"inference_bundle.json")
    completion = core.read(run/"completion_manifest.json")["files"]
    if digest(run/"inference_bundle.json") != completion["inference_bundle.json"]:
        raise ValueError("Inference bundle changed.")
    if bundle["environment"] != core.environment():
        raise ValueError("Use the pinned training environment.")
    spec = bundle["feature_spec"]
    names = spec["descriptors"]
    if (names not in ([], list(DESCRIPTORS8)) or spec["n_bits"] != N_BITS
            or bool(names) != bundle["use_descriptors"]):
        raise ValueError("Unsupported descriptor order.")
    if spec.get("rdkit_version", rdBase.rdkitVersion) != rdBase.rdkitVersion:
        raise ValueError("RDKit version differs.")
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or len(Chem.GetMolFrags(mol)) != 1:
        raise ValueError("Input must be one valid connected molecule.")
    generator = rdFingerprintGenerator.GetMorganGenerator(
        radius=spec["radius"], fpSize=spec["n_bits"], includeChirality=spec["include_chirality"])
    x = np.concatenate([generator.GetFingerprintAsNumPy(mol),
                        [getattr(Descriptors,name)(mol) for name in names]]).astype(np.float32)[None,:]
    if not np.isfinite(x).all() or x.shape[1] != spec["dimensions"]["input"]:
        raise ValueError("Feature schema mismatch.")
    output = []
    for label in bundle["labels"]:
        entry = bundle["models"][label]
        path = run/bundle["models_directory"]/entry["file"]
        if digest(path) != entry["sha256"]:
            raise ValueError("Model checksum changed.")
        model = joblib.load(path)
        if model.n_features_in_ != x.shape[1]:
            raise ValueError("Model expects a different feature count.")
        # The ndarray feature order is frozen and checked above.
        import warnings
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="X does not have valid feature names, but LGBMClassifier was fitted with feature names", category=UserWarning)
            score = float(model.predict_proba(x)[0,1])
        output.append({"label":label, "score":score, "predicted":score>=bundle["primary_threshold"]})
    return {"smiles":Chem.MolToSmiles(mol), "candidate":bundle["selected_candidate"],
            "descriptors":len(names), "threshold":bundle["primary_threshold"],
            "score_meaning":"uncalibrated model score, not aroma intensity",
            "labels":sorted(output,key=lambda row:row["score"],reverse=True)}

```

### Jalankan prediksi nyata

Contoh SMILES adalah satu molekul vanilin. Classifier menerima fitur RDKit, bukan teks mentah. Tabel menampilkan 15 teratas, tetapi hasil memuat 109 label. Skor × 100 hanya format tampilan; jumlahnya tidak harus 100%.

In [7]:
SMILES = "COc1cc(C=O)ccc1O"
hasil = inference.predict(run, SMILES)
tabel = pd.DataFrame(hasil["labels"])
tabel["skor_tampilan_persen"] = tabel["score"] * 100
assert len(tabel) == len(bundle["labels"])
assert set(tabel.label) == set(bundle["labels"])
assert (tabel.predicted == (tabel.score >= hasil["threshold"])).all()
print({"SMILES_kanonis": hasil["smiles"], "kandidat": hasil["candidate"],
       "ambang": hasil["threshold"], "label_positif": int(tabel.predicted.sum()),
       "makna_skor": hasil["score_meaning"]})
display(tabel.head(15))

{'SMILES_kanonis': 'COc1cc(C=O)ccc1O', 'kandidat': 'lgbm_D', 'ambang': 0.5, 'label_positif': 16, 'makna_skor': 'uncalibrated model score, not aroma intensity'}


,label,score,predicted,skor_tampilan_persen
0,vanilla,0.993578,True,99.357774
1,gourmand,0.964123,True,96.412263
2,caramel,0.963594,True,96.359382
3,creamy,0.945421,True,94.542109
4,chocolate,0.934045,True,93.404466
5,powdery,0.913262,True,91.326208
6,phenolic,0.894484,True,89.448405
7,sweet,0.868036,True,86.803637
8,balsamic,0.861837,True,86.183691
9,spicy,0.828794,True,82.879361


### Lihat keputusan positif

Daftar bisa berisi beberapa, satu, atau tanpa label positif. Top 15 sebelumnya membatasi tampilan, bukan aturan prediksi.

In [8]:
display(tabel.loc[tabel.predicted, ["label", "score", "skor_tampilan_persen"]])

,label,score,skor_tampilan_persen
0,vanilla,0.993578,99.357774
1,gourmand,0.964123,96.412263
2,caramel,0.963594,96.359382
3,creamy,0.945421,94.542109
4,chocolate,0.934045,93.404466
5,powdery,0.913262,91.326208
6,phenolic,0.894484,89.448405
7,sweet,0.868036,86.803637
8,balsamic,0.861837,86.183691
9,spicy,0.828794,82.879361


### Periksa input di luar kontrak

CCO.O mempunyai dua fragmen dan seharusnya ditolak. Ini pemeriksaan kebijakan input, bukan prediksi campuran.

In [9]:
try:
    inference.predict(run, "CCO.O")
except ValueError as error:
    assert str(error) == "Input must be one valid connected molecule."
    print(type(error).__name__ + ": " + str(error))
else:
    raise AssertionError("Input multifragmen seharusnya ditolak.")

ValueError: Input must be one valid connected molecule.


### `inference.main` — kode aktual

**File:** `KODE_ML/03_Kode/inference.py`.

**Input:** argumen terminal --smiles dan opsional --run. **Output:** JSON dari predict(). Tanpa --run, main_run() memilih alamat utama. CLI memakai fungsi yang sama dengan notebook sehingga tidak ada jalur inferensi kedua.

Sel berikut menampilkan definisi lengkap beserta lokasi dan hash sumber. Isinya dipakai oleh pipeline, bukan implementasi kedua.

In [10]:
tampilkan_sumber(inference.main)

**Source aktual:** `KODE_ML/03_Kode/inference.py`, baris 61–67. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `a9cdd7d49ed7a308134c9d6b82b47f1cbfcd720a9c58d02511ecf78a9b432a4a`.

```python
def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--run",type=Path,default=None)
    parser.add_argument("--smiles",required=True)
    args = parser.parse_args()
    from .results import main_run
    print(json.dumps(predict(args.run or main_run(),args.smiles),indent=2))

```

## Bekal sidang

- **Input classifier apa?** Array satu baris dengan urutan fitur terkunci; RDKit mengubah SMILES menjadi array itu.
- **Mengapa skor tidak berjumlah 100%?** Tiap label punya classifier biner sendiri; beberapa label dapat positif sekaligus.
- **Bisa prediksi campuran?** Pipeline ini hanya untuk satu molekul terhubung dan belum dilatih memakai target campuran/proposinya.
- **Apakah ini rekomendasi formula/intensitas?** Bukan. Ini profil label aroma molekul sesuai anotasi training.
- **Apakah contoh vanilin membuktikan aroma secara sensoris?** Tidak; ini inferensi model, bukan pengukuran laboratorium.

Alur belajar selesai: sumber → cleaning/label → fitur → split → classifier → tuning/CV → evaluasi → inferensi. Source aktual tetap satu di **KODE_ML/03_Kode/**.